# SETTTINGS

In [34]:
import warnings
warnings.filterwarnings('ignore')

In [35]:
# DataSet
dataset = "iris.csv"

# Outliers Switch & Proportion
remove_outliers = False
outlier_proportion = 1.5

# Feature Selection Embedded
rf_feature_embedded = False

# Columns to Drop
columns_to_drop = []

# Target Variable Assignment
target_name = 'species'

# Data Types & Conversion
data_types = {0:'int32', 1:'int64', 2:'float32', 3:'float64', 4:'str', 5:'object'}
features_to_convert = {}

# Model Name
pickle_name = "iris_multi_classify_model_1"

<hr style='height:10px'>

<br id='basic'>

## BASIC LIBRARIES

* <b>Numpy</b> - Numerical Analysis Library
* <b>Pandas</b> - For Data Load, Exploratory Data Analysis and Data Preprocessing
* <b>Matplotlib</b> - For Charts or Plots
* <b>Seaborn</b> - for Charts or Plots built on top of Matplotlib


In [36]:
# Numpy
import numpy as np

# Pandas
import pandas as pd

# Matplotlib
import matplotlib.pyplot as plt

# Seaborn
import seaborn as sns

from scipy.stats import skew

import statistics as st

In [37]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

<hr style='height:10px'>

<br id='preparation'>

## DATA PREPARATION

In [38]:
df = pd.read_csv('/content/'+dataset)

## Strip Trail Spacesin Column Names

In [39]:
df.columns = [col.strip() for col in df.columns]

In [40]:
df.dtypes

,0
sepal_length,float64
sepal_width,float64
petal_length,float64
petal_width,float64
species,object


<hr style='height:10px'>

<br id='eda'>

## EXPLORATORY DATA ANALYSIS

<br id='shape'>

#### DataFrame Shape

In [41]:
df.shape   # (rows, columns)

(150, 5)

<br id='head'>

#### First 5 Rows

In [42]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


<br id='tail'>

#### Last 5 Rows

In [43]:
df.tail()

,sepal_length,sepal_width,petal_length,petal_width,species
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica
149,5.9,3.0,5.1,1.8,virginica


<br id='freq'>

#### Target Variable - Frequency

In [44]:
df[target_name].value_counts()

,count
species,
setosa,50
versicolor,50
virginica,50


<br id='dtypes'>

#### DataTypes

In [45]:
df.dtypes

,0
sepal_length,float64
sepal_width,float64
petal_length,float64
petal_width,float64
species,object


<br id='cat'>

#### Categorical Features

In [46]:
cat_features = df.dtypes[df.dtypes == 'object'].index

<br id='catstat'>

#### Summary Statistics of all Categorical Fetures

* We have used Pandas describe method on all the Numerical Features to understand Statistical Information like -
    
        1. Max
        2. Min

In [47]:
# Summarize categorical features
df[cat_features].describe()

,species
count,150
unique,3
top,setosa
freq,50


<br id='num'>

###  Numerical Features

In [48]:
for features in df.dtypes[df.dtypes != 'object'].index:
    print(features)

sepal_length
sepal_width
petal_length
petal_width


<br id='numstat'>

### Summary Statistics of all Numerical Fetures

* We have used Pandas describe method on all the Numerical Features to understand Statistical Information like -
    
        1. Mean
        2. Standard Deviation
        3. Max
        4. Min
        5. 25th Percentile
        6. 50th Percentile
        7. 75th Percentile

In [49]:
num_features = df.dtypes[df.dtypes != 'object'].index

In [50]:
# Summarize numerical features
df[num_features].describe()

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


<br id='cat'>

### List of Categorical Features in DataFrame

In [51]:
df.dtypes[df.dtypes=='object']

,0
species,object


<hr style='height:10px'>

<br id='neg'>

### Replace Negative Values with Zero

* Replaced Negative Values as Multinomial Naive Bayes and related Algorithms doesn't accept Negative Values

In [52]:
num = df._get_numeric_data()

num[num < 0] = 0

<br id='unnec'>

### Remove Unnecessary Features

In [53]:
df = df.drop(columns_to_drop, axis=1)

<br id='dropdups'>

### Drop Duplicates

1. Check the Shape before removing duplicates
2. Used Pandas "drop_duplicates" method to remove duplicates
3. Cross checked Shape after removing duplicates to understand how many duplicate rows present in the dataframe
4. We have found 33 duplicate rows in the given sample of around 200K rows
5. Duplicate rows are subject to change as we have taken Sample of 200K rows rather than all to reduce overhead on the server during Training Phase

In [54]:
# Check the Number of Rows before Removing Duplicates
df.shape

(150, 5)

In [55]:
# Call drop_duplicates on DataFrame to remove Duplicates and Assign it back to DataFrame
df = df.drop_duplicates()

In [56]:
# Check the Number of Rows after Removing Duplicates
df.shape

(149, 5)

<br id='inf'>

### Replace "Inf" Values with it's mean value

In [57]:
df = df.replace([np.inf, -np.inf], np.nan)   # replaced "Inf" and "-Inf" with "NaN"

### Remove Unwanted OUTLIERS !!!

### Get Feature and respective outliers count

In [58]:
for feature in df.dtypes[df.dtypes != 'object'].index:
    count = []
    median = np.median(df[feature])
    upper_quartile = np.percentile(df[feature], 75)
    lower_quartile = np.percentile(df[feature], 25)
    iqr = upper_quartile - lower_quartile
    maxim = df[df[feature] <= upper_quartile + outlier_proportion * iqr][feature].max()
    minim = df[df[feature] >= lower_quartile - outlier_proportion * iqr][feature].min()
    for val in df[feature].values:
        if((val > maxim) or (val < minim)):
            count.append(val)
    if(len(count) > 0):
        print(feature, 'has ', str(len(count)), ' outliers. The Outliers are', ' : ', count)
        print('\n')

sepal_width has  4  outliers. The Outliers are  :  [np.float64(4.4), np.float64(4.1), np.float64(4.2), np.float64(2.0)]




### Remove all the Outliers

In [59]:
if remove_outliers:
    for feature in df.dtypes[df.dtypes!='object'].index:
        count = []
        median = np.median(df[feature])
        upper_quartile = np.percentile(df[feature], 75)
        lower_quartile = np.percentile(df[feature], 25)
        iqr = upper_quartile - lower_quartile
        maxim = df[df[feature] <= upper_quartile + outlier_proportion * iqr][feature].max()
        minim = df[df[feature] >= lower_quartile - outlier_proportion * iqr][feature].min()
        for val in df[feature].values:
            if((val > maxim) or (val < minim)):
                df[feature] = df[df[feature] != val][feature]

### Check the Outliers again

In [60]:
for feature in df.dtypes[df.dtypes!='object'].index:
    count = []
    median = np.median(df[feature])
    upper_quartile = np.percentile(df[feature], 75)
    lower_quartile = np.percentile(df[feature], 25)
    iqr = upper_quartile - lower_quartile
    maxim = df[df[feature] <= upper_quartile + outlier_proportion * iqr][feature].max()
    minim = df[df[feature] >= lower_quartile - outlier_proportion * iqr][feature].min()
    for val in df[feature].values:
        if((val > maxim) or (val < minim)):
            count.append(val)
    if(len(count) > 0):
        print(feature, 'has ', str(len(count)), ' outliers. The Outliers are', ' : ', count)
        print('\n')

sepal_width has  4  outliers. The Outliers are  :  [np.float64(4.4), np.float64(4.1), np.float64(4.2), np.float64(2.0)]




<hr style='height:4px'>

### Remove NULLs

In [61]:
# Check Columns that has NULLs
df.isnull().sum()

,0
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species,0


### NULLs

#### Fill all the NULLs in Numerical Features with fill (when Values in Numerical Features are Repetetive in Nature)

In [62]:
df.isnull().sum()

,0
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species,0


In [63]:
for feature in df.isnull().sum()[df[df.dtypes.index].isnull().sum() > 0].index:
    if (np.dtype(df[feature]) == np.dtype('int64') or np.dtype(df[feature]) == np.dtype('float64')):
        if skew(df[feature]) <= 5.0:
            df[feature] = df[feature].mean()
        elif skew(df[feature]) >= 5.0 and skew(df[feature]) <= 10.0:
            df[feature] = df[feature].ffill()
            df[feature] = df[feature].bfill()
        else:
            df[feature] = df[feature].fillna(0)
    else:
        if len(df[feature].unique()) / len(df[feature]) <= 5.0:
            df[feature] = df[feature].fillna(st.mode(df[df[feature].notnull()][feature]))
        elif len(df[feature].unique()) / len(df[feature]) >= 5.0 and len(df[feature].unique()) / len(df[feature]) <= 10.0:
            df[feature] = df[feature].ffill()
            df[feature] = df[feature].bfill()
        else:
            df[feature] = df[feature].fillna('Missing')

#### Check NULLs again

In [64]:
# Check Columns that has NULLs After dealing with NULL values in the respective Features
df.isnull().sum()

,0
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species,0


## Convert Data Types of Features

In [65]:
if len(features_to_convert) != 0:
    for col, data_type in features_to_convert.items():
        df[col] = df[col].astype(data_type)

### Label Encoding

In [66]:
final_dict = {}

In [67]:
from sklearn.preprocessing import LabelEncoder

In [68]:
for feature in df.dtypes[df.dtypes != 'object'].index:
    final_dict[feature] = [df[feature].dtype, min(df[feature]), max(df[feature]), np.mean(df[feature])]

In [69]:
for feature in df.dtypes[df.dtypes=='object'].index:
    label_encoder  = LabelEncoder()
    feature_unique_values = list(df[feature].unique())
    df[feature] = label_encoder.fit_transform(df[feature])
    final_dict[feature] = ['object', feature_unique_values, label_encoder]

In [70]:
final_dict['target'] = target_name

<hr style='height:10px'>

<br id='model'>

## Model Preparation & Training

<br id='targetinput'>

### Seperate objects for TARGET & INPUT features

In [71]:
# Separate object for target feature
y = df[target_name]

# Separate Object for Input Features
X = df.drop(target_name, axis=1)

<br id='split'>

### Train Test Split

* We have used train_test_split from sklearn (sklearn.model_selection)
* We have choosen Test Size as 30%. So, the Train size is going to be 70%
* X_train - Train set of Independent Variables
* X_test -  Test set of Independent Variables
* y_train - Train set of Dependent/Target Variable
* y_test  - Test set of Dependent/Target Variable

In [72]:
from sklearn.model_selection import train_test_split

In [73]:
# Split X and y into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)

<hr style='height:10px'>

<br id='features'>

## Feature Selection
    Filter Methods
         Basic
             Constant Features
             Quasi-Constant Features
             Duplicated Features
         Correlation

In [74]:
X_train.shape, X_test.shape

((104, 4), (45, 4))

In [75]:
# I keep a copy of the dataset with all the variables
# to measure the performance of machine learning models
# at the end of the notebook

X_train_original = X_train.copy()
X_test_original = X_test.copy()

In [76]:
X_train_original.head()

,sepal_length,sepal_width,petal_length,petal_width
9,4.9,3.1,1.5,0.1
105,7.6,3.0,6.6,2.1
143,6.8,3.2,5.9,2.3
126,6.2,2.8,4.8,1.8
117,7.7,3.8,6.7,2.2


### Remove constant & quasi-constant features

* Import VarianceThreshold (sklearn.feature_selection)
* Assign the threshold 0.01, which indicates 1% variance i.e., 99% constant
* Get all the features that are not Constant (100% Constant) or Quasi-Constant (99% or more Constant)
* Store all the collected Non-Constant features in "feature_to_keep" variable (List)
* Select only those features that are in "feature_to_keep" variable in X_train and X_test

In [77]:
from sklearn.feature_selection import VarianceThreshold

In [78]:
# remove quasi-constant features
sel = VarianceThreshold(
    threshold=0.01)  # 0.01 indicates 99% of observations approximately

sel.fit(X_train)  # fit finds the features with low variance

sum(sel.get_support()) # how many not quasi-constant?

np.int64(4)

In [79]:
features_to_keep = X_train.columns[sel.get_support()]

In [80]:
features_to_keep

Index(['sepal_length', 'sepal_width', 'petal_length', 'petal_width'], dtype='object')

In [81]:
# we can then remove the features like this
X_train = sel.transform(X_train)
X_test = sel.transform(X_test)

X_train.shape, X_test.shape

((104, 4), (45, 4))

In [82]:
# sklearn transformations lead to numpy arrays
# here I transform the arrays back to dataframes
# please be mindful of getting the columns assigned
# correctly

X_train= pd.DataFrame(X_train)
X_train.columns = features_to_keep

X_test= pd.DataFrame(X_test)
X_test.columns = features_to_keep

### Remove duplicated features

* Create a loop such that every Features maps with all other features available
* Check every combination of features equal or not
* If it is equal/duplicate, add to list "duplicated_feat"
* Remove duplicated features using drop method

In [83]:
# check for duplicated features in the training set
duplicated_feat = []
for i in range(0, len(X_train.columns)):
    col_1 = X_train.columns[i]

    for col_2 in X_train.columns[i + 1:]:
        if X_train[col_1].equals(X_train[col_2]):
            duplicated_feat.append(col_2)

len(duplicated_feat)

0

In [84]:
# remove duplicated features
X_train.drop(labels=duplicated_feat, axis=1, inplace=True)
X_test.drop(labels=duplicated_feat, axis=1, inplace=True)

X_train.shape, X_test.shape

((104, 4), (45, 4))

In [85]:
X_train_basic = X_train.copy()
X_test_basic = X_test.copy()

In [86]:
X_train.shape

(104, 4)

### Remove correlated features

* Get Correlation Mapping between all the Features using corr()
* Loop all the Features such that every Feature maps with other Features available
* Check Correlation between every combination of features greater than given "threshold"
* Store all the Correlated features greater than threshold in a set
* Return the set
* Drop the Correlated features from X_train and X_test (Input/Independent Variables)

In [87]:
# find and remove correlated features
def correlation(dataset, threshold):
    col_corr = set()  # Set of all the names of correlated columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold: # we are interested in absolute coeff value
                colname = corr_matrix.columns[i]  # getting the name of column
                col_corr.add(colname)
    return col_corr

In [88]:
corr_features = correlation(X_train, 0.8)
print('correlated features: ', len(set(corr_features)))

correlated features:  2


In [89]:
X_train.drop(labels=corr_features, axis=1, inplace=True)
X_test.drop(labels=corr_features, axis=1, inplace=True)

X_train.shape, X_test.shape

((104, 2), (45, 2))

In [90]:
X_train_corr = X_train.copy()
X_test_corr = X_test.copy()

In [91]:
X_train.shape

(104, 2)

### Select features by random forests derived importance

In [92]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

In [93]:
sel_ = SelectFromModel(RandomForestClassifier(n_estimators=400))
sel_.fit(X_train, y_train)

SelectFromModel(estimator=RandomForestClassifier(n_estimators=400))

In [94]:
X_train_rf = pd.DataFrame(sel_.transform(X_train))
X_test_rf = pd.DataFrame(sel_.transform(X_test))

In [95]:
# add the columns name
X_train_rf.columns = X_train.columns[(sel_.get_support())]
X_test_rf.columns = X_train.columns[(sel_.get_support())]

In [96]:
X_train_rf.shape, X_test_rf.shape

((104, 1), (45, 1))

In [97]:
rf_features = X_train_rf.columns

In [98]:
final_dict['rf_features'] = rf_features

In [99]:
if rf_feature_embedded:
    X_train = X_train_rf
    X_test = X_test_rf

<hr style='height:10px'>

<br id='advlibraries'>

## CLASSIFICATION ALGORITHMS

* Random Forest Classifier
* Gradient Boosting Classifier
* Logistic Regression
* Decision Tree Classifier
* K-Neighbors Classifier
* Multinomial Naive Bayes
* Gaussian Naive Bayes
* Benouli Naive Bayes

#### Random Forest Classifier

In [100]:
from sklearn.ensemble import RandomForestClassifier

#### Gradient Boosting Classifier

In [101]:
from sklearn.ensemble import GradientBoostingClassifier

#### Logistic Regression Classifier

In [102]:
from sklearn.linear_model import LogisticRegression

#### Decision Tree Classifier

In [103]:
from sklearn.tree import DecisionTreeClassifier

#### K-Nearest Neighbors Classifier

In [104]:
from sklearn.neighbors import KNeighborsClassifier

#### Naive Bayes Classifiers

In [105]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB

In [106]:
from sklearn.pipeline import make_pipeline

#### Standard Scaler

In [107]:
from sklearn.preprocessing import StandardScaler

### Pipeline

In [108]:
pipeline_dict = {'lgl2': make_pipeline(StandardScaler(), LogisticRegression(penalty='l2', random_state=123)),
                 'rfc': make_pipeline(StandardScaler(), RandomForestClassifier(random_state=123)),
                 'gbc': make_pipeline(StandardScaler(), GradientBoostingClassifier(random_state=123)),
                 'dtc': DecisionTreeClassifier(random_state=123),
                 'mnb': MultinomialNB(),
                 'gnb': GaussianNB(),
                 'bnb': BernoulliNB()
                }

<hr style='height:10px'>

<br id='hyper'>

## Hyperparameters

#### Logistic Regression Hyperparameters

* logisticregression__C -->  parameter is like L1 Penalty. It helps in reducing noise. Less the C values, more the Penalty.

In [109]:
# Logistic Regression hyperparameters
lgl2_hyperparameters = { 'logisticregression__C': np.linspace(1e-3, 1e3, 10)}

#### Random Forest Hyperparameters

* randomforestclassifier__n_estimators --> Maximum Number of Trees
* randomforestclassifier__max_features --> Maximum number of features to be selected during Model training phase

In [110]:
# Random Forest hyperparameters
rfc_hyperparameters = { 'randomforestclassifier__n_estimators': [100, 200],
                         'randomforestclassifier__max_features': ['auto', 'sqrt', 0.33]
                     }

#### Gradient Boosting Hyperparameters

* gradientboostingclassifier__n_estimators  --> Maximum number of Trees
* gradientboostingclassifier__learning_rate --> Learning rate is a tuning parameter in an optimization algorithm that determines the step size at each iteration while moving toward a minimum of a loss function.
* gradientboostingclassifier__max_depth     --> Depth of a Decision Tree

In [111]:
# Boosted Tree hyperparameters
gbc_hyperparameters = {'gradientboostingclassifier__n_estimators': [100, 200],
                    'gradientboostingclassifier__learning_rate': [0.05, 0.1, 0.2],
                     'gradientboostingclassifier__max_depth': [1, 3, 5]
                     }

#### Decision Tree Hyperparameters

* criterion --> Classification Purity (Gini, Entropy)
* splitter  --> The strategy used to choose the split at each node. Supported strategies are “best” to choose the best split and “random” to choose the best random split.
* min_samples_leaf --> The minimum number of samples required to be at a leaf node
* min_samples_split -->  The minimum number of samples required to split an internal node
* max_depth --> Depth of a Decision Tree
* random_state --> Value to be set to distribute same set of randomized rows to all the algorithms

In [112]:
# Decision Tree Classifier hyperparameters
dtc_hyperparameters = {'criterion': ['gini','entropy'],
                      'splitter': ['best', 'random'],
                      'min_samples_leaf': [1,3,5,7, 10],
                      'min_samples_split': [2,5,7,10,12],
                      'max_depth': [2,4,6,8,10,None],
                      'random_state': [0]
                      }

#### K-Nearest Neighbors Hyperparameters

* p           -->  Experiment Error Proportion
* n_neighbors --> Number of neighbors to use by default for kneighbors queries.

##### metric (KNN)

###### City Block

![city_block_distance.png](attachment:city_block_distance.png)

###### Euclidean Distance

<font size=4><center>sqrt(sum((x - y)^2))<center></font>

In [113]:
# K Nearest Neighbors hyperparameters
knn_hyperparameters = {'p': [1, 2, 3, 4, 5],
                      'n_neighbors': list(range(1,30)),
                      'metric': ["euclidean", "cityblock"]
                      }

#### Multinomial Naive Bayes Hyperparameters

* alpha     --> Additive (Laplace/Lidstone) smoothing parameter (0 for no smoothing).

* fit_prior --> Whether to learn class prior probabilities or not. If false, a uniform prior will be used.

In [114]:
# Multinomial Naive Bayes Hyperparameters
mnb_hyperparameters = { 'alpha': np.linspace(0.5, 1.5, 6),
                       'fit_prior': [True, False]
                        }

#### Gaussian Naive Bayes Hyperparameters

* fit_prior --> Whether to learn class prior probabilities or not. If false, a uniform prior will be used.

#### Binomial Naive Bayes Hyperparameters

* alpha     --> Additive (Laplace/Lidstone) smoothing parameter (0 for no smoothing).

* fit_prior --> Whether to learn class prior probabilities or not. If false, a uniform prior will be used.

In [115]:
bnb_hyperpaeters = {'alpha': np.linspace(0.5, 1.5, 6),
                       'fit_prior': [True, False]
                   }

#### Hyperparameters Dictionary

* Hyperparameter Dictionary going to be passed as Parameter to GridSearchCV
* It simplifies in applying hyperparameters to all the respective algorithms in GridSearchCV

In [116]:
# Create hyperparameters dictionary
hyperparameters = {
           'lgl2': lgl2_hyperparameters,
           'rfc': rfc_hyperparameters,
           'gbc': gbc_hyperparameters,
           'dtc': dtc_hyperparameters,
           'mnb': mnb_hyperparameters,
           'bnb': bnb_hyperpaeters,
           'gnb': {}
          }

<hr style='height:10px'>

<br id='fit'>

### Training / Fitting

* Import GridSearchCV for Cross Validation
* Create a dictionary to store fitted models - fitted_models
* Loop pipeline_dict to get Pipeline and Name of Algorithms to feed GridSearchCV
* Fit the Model with X_train and y_train
* Store the Model in fitted_models dictionary
* Print once the model been fitted and stored in fitted_models dictionary
* Check the availability of all the Cross-Validation Objects
* Check All the Algorithms are fitted

In [117]:
from sklearn.model_selection import GridSearchCV

In [118]:
fitted_models = {}

for name, pipeline in pipeline_dict.items():

    model = GridSearchCV(pipeline, hyperparameters[name], cv=10, n_jobs=-1)

    model.fit(X_train, y_train)

    fitted_models[name] = model

    print(name, 'has been fitted.')

lgl2 has been fitted.
rfc has been fitted.
gbc has been fitted.
dtc has been fitted.
mnb has been fitted.
gnb has been fitted.
bnb has been fitted.


<hr style='height:10px'>

<br id='eval'>

## EVALUATE MODELS

### Predict Values by Best Score & Accuracy Score of Fitted Models

In [119]:
pred = {}
for name, model in fitted_models.items():
    pred[name] = fitted_models[name].predict(X_test)

#### Display first 10 predictions

In [120]:
for key, val in pred.items():
    print('First 10 {} predicted values are {}'.format(key, pred[key][:10]))

First 10 lgl2 predicted values are [1 1 2 0 1 0 0 0 1 1]
First 10 rfc predicted values are [1 1 1 0 1 0 0 0 1 1]
First 10 gbc predicted values are [1 2 2 0 0 0 0 0 0 1]
First 10 dtc predicted values are [1 2 2 0 1 0 0 0 1 1]
First 10 mnb predicted values are [1 1 1 0 1 0 0 0 1 1]
First 10 gnb predicted values are [1 1 2 0 1 0 0 0 1 1]
First 10 bnb predicted values are [2 2 2 2 2 2 2 2 2 2]


### Accuracy Score

In [121]:
from sklearn.metrics import accuracy_score

In [122]:
final_dict['model_score'] = []
i = 0
for key, val in pred.items():
    acc_score = accuracy_score(y_test, pred[key])
    print('{} accuracy score is {}'.format(key, acc_score))
    final_dict['model_score'].append((i, key, acc_score))
    i += 1

lgl2 accuracy score is 0.8444444444444444
rfc accuracy score is 0.6666666666666666
gbc accuracy score is 0.6888888888888889
dtc accuracy score is 0.6888888888888889
mnb accuracy score is 0.7111111111111111
gnb accuracy score is 0.8666666666666667
bnb accuracy score is 0.24444444444444444


### Confusion Matrix

In [123]:
# Import confusion_matrix
from sklearn.metrics import confusion_matrix

In [124]:
# Display confusion matrix for y_test and pred
for key, val in pred.items():
    print ('{} confusion matrix is \n{}\n'.format(key, confusion_matrix(y_test, pred[key])))

lgl2 confusion matrix is 
[[16  0  0]
 [ 0 13  5]
 [ 0  2  9]]

rfc confusion matrix is 
[[16  0  0]
 [ 0 10  8]
 [ 0  7  4]]

gbc confusion matrix is 
[[16  0  0]
 [ 2  6 10]
 [ 0  2  9]]

dtc confusion matrix is 
[[15  1  0]
 [ 0  9  9]
 [ 0  4  7]]

mnb confusion matrix is 
[[16  0  0]
 [ 0 10  8]
 [ 0  5  6]]

gnb confusion matrix is 
[[16  0  0]
 [ 0 14  4]
 [ 0  2  9]]

bnb confusion matrix is 
[[ 0  0 16]
 [ 0  0 18]
 [ 0  0 11]]



## Classification Report

In [125]:
from sklearn.metrics import classification_report

In [126]:
for key, val in pred.items():
    print('\n{} classification report is: \n {}'.format(key, classification_report(y_test, pred[key])))
    print('---------------------------------------------------------------------------------------------')


lgl2 classification report is: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       0.87      0.72      0.79        18
           2       0.64      0.82      0.72        11

    accuracy                           0.84        45
   macro avg       0.84      0.85      0.84        45
weighted avg       0.86      0.84      0.85        45

---------------------------------------------------------------------------------------------

rfc classification report is: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       0.59      0.56      0.57        18
           2       0.33      0.36      0.35        11

    accuracy                           0.67        45
   macro avg       0.64      0.64      0.64        45
weighted avg       0.67      0.67      0.67        45

-----------------------------------------------------------------------------

<hr style='height:10px'>

<br id='finalmodel'>

## Best Estimator & Tuned Parameters

In [127]:
best_model = final_dict['model_score'][max(final_dict['model_score'], key = lambda i : i[2])[0]][1]

### Save the Model

In [128]:
import pickle

In [129]:
final_dict['model'] = fitted_models[best_model].best_estimator_

In [132]:
with open('/content/'+pickle_name + '.pkl', 'wb') as f:
    pickle.dump(final_dict, f)

In [133]:
fitted_models[best_model].best_estimator_

GaussianNB()

<hr style='height:10px'>